In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the output directory exists before any savefig/to_csv calls in this notebook.
os.makedirs('output', exist_ok=True)

**Exploring Data**

In [ ]:
df = pd.read_csv('data/bank_transactions.csv')

Every row represents a unique transaction, along with the information of the user who performed it.

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

As we can see, all of the min values are 0. But since we are going to make RFM analysis, we will only analyze the TransactionAmount (INR) column, which is the most important column for RFM analysis. So we will check the min and max values of this column in future. But first let's check amount of missing values.

In [ ]:
df.isnull().sum()

**Data Preprocessing**

In [ ]:
# We only drop rows with missing values in columns critical for RFM analysis.
# This prevents losing valid transactions due to missing non-essential data (e.g., location or gender)
df.dropna(subset=['CustomerID', 'TransactionDate', 'TransactionAmount (INR)'], inplace=True)

df.isnull().sum()

In [ ]:
df.info()

In [ ]:
duplicates = df.duplicated(subset=['CustomerID', 'TransactionAmount (INR)', 'TransactionDate', 'TransactionTime']).sum()
print(duplicates)

There is no duplicate transaction in the dataset, so we can continue.

In [ ]:
date_columns = ['TransactionDate']
for columns in date_columns:
    df[columns] = pd.to_datetime(df[columns], format='%d/%m/%y', errors='coerce')

# errors='coerce' silently turns any string that doesn't match the expected format into NaT.
# We explicitly check for that here instead of assuming the conversion was clean.
n_bad_dates = df['TransactionDate'].isnull().sum()
print(f"Rows with unparseable TransactionDate (now NaT): {n_bad_dates}")

if n_bad_dates > 0:
    df = df.dropna(subset=['TransactionDate'])
    print(f"Dropped {n_bad_dates} rows with invalid TransactionDate.")

print(df[date_columns])

In [ ]:
df.info()

Data type has been successfully converted. `TransactionDate` is now in datetime format, allowing for time-based calculations. We also verified that no rows were silently corrupted by the conversion (`errors='coerce'` would otherwise hide invalid dates as `NaT`).

**Note:** `CustomerDOB` is not used in the RFM calculation (Recency, Frequency, and Monetary are derived solely from `TransactionDate`, `CustomerID`, and `TransactionAmount`), so we drop it from further analysis rather than spending time fixing its formatting issues.

In [ ]:
df = df.drop(columns=['CustomerDOB'])

## Outlier Analysis

This part won't be used in RFM Segmentation. It's only for visualization purposes.

In [ ]:
df.describe()['TransactionAmount (INR)'].round(2)

Lets look at min and max rows of the dataset to see if there are any outliers in the transaction amount.

As we can see, maximum transaction amount is way above the average transaction amount, which indicates that there are some outliers in the dataset. We will handle these outliers in the next step. Lets check the amount of transactions with a zero amount first.

In [ ]:
(df['TransactionAmount (INR)'] == 0).sum()

There are 835 transactions with a zero amount. These could be due to card authorization holds for free trials, security verifications, or transaction cancellations. Since the core objective of RFM analysis is to measure actual financial value (Monetary) and revenue-generating frequency, keeping these records would artificially inflate customer engagement scores without contributing to revenue. Therefore, we will drop these records to ensure the accuracy and reliability of our segments.

In [ ]:
df = df[df['TransactionAmount (INR)'] > 0]

Now its time to analyze the max values. As we can see max value is way above the average value, which indicates that there are some outliers in the dataset. We will take a look at these outliers in the next step.

We use the 99th percentile to cap outliers in `TransactionAmount (INR)` solely for visualization and distribution analysis purposes. The mean (1574 INR) is far below the max (1.56M INR), indicating a heavily right-skewed distribution. Capping at the 99th percentile helps us visualize the distribution of 99% of the data without the distortion of extreme outliers. However, we will retain and use the original, uncapped values for the actual RFM Monetary calculation to preserve the true financial value of our top customers.

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=df, x='TransactionDate', y='TransactionAmount (INR)', estimator='sum', errorbar=None)
plt.title('Total Transaction Amount Over Time')
plt.xlabel('Date')
plt.ylabel('Total Transaction Amount (INR)')
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=df, y='TransactionAmount (INR)')

In [ ]:
upper_limit = df['TransactionAmount (INR)'].quantile(0.99)
df_capped = df.copy()  # Creating a copy of the original DataFrame to use uncapped values for RFM segmentation.
df_capped.loc[df['TransactionAmount (INR)'] > upper_limit, 'TransactionAmount (INR)'] = upper_limit

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='TransactionAmount (INR)', bins=50, kde=False, ax=axes[0], color='lightblue',
             binrange=(0, 25000))  # kde value is set to False due to potential misinterpretation.
axes[0].set_title('TransactionAmount (INR) - Before Capping')
axes[0].set_xlabel('Amount (INR)')
axes[0].set_xlim(0, 25000)
axes[0].set_ylim(0, 550000)

sns.histplot(data=df_capped, x='TransactionAmount (INR)', bins=50, kde=True, ax=axes[1], color='green', binrange=(0, 25000))
axes[1].set_title('TransactionAmount (INR) - After Capping')
axes[1].set_xlabel('Amount (INR)')
axes[1].set_xlim(0, 25000)
axes[1].set_ylim(0, 550000)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=df_capped, x='TransactionDate', y='TransactionAmount (INR)', estimator='sum', errorbar=None)
plt.title('Total Transaction Amount Over Time')
plt.xlabel('Date')
plt.ylabel('Total Transaction Amount (INR)')
plt.tight_layout()
plt.show()

As we can see, although this chart closely resembles the one we generated first, there is 
a noticeable decrease in the peaks. For example, the peak around August 5th drops from 
approximately 47M INR to 40M INR after capping — a reduction of roughly 15%. The overall 
shape and timing of the peaks and dips remain unchanged, confirming that capping mainly 
affects days with a concentration of extreme transactions, without distorting the 
underlying temporal trend of the data.

## Why is there no visible spike before capping?

Before capping, values above 20,000 INR make up only ~1% of the transactions, and — more 
importantly — this 1% is spread across a wide range (from just above 20,000 up to 
1,560,034 INR), rather than concentrated in one place. Since the x-axis is limited to 
25,000 for readability, most of this spread (25,000 to 1.56M) falls outside the visible 
range entirely.

After capping, every value above the 99th percentile (20,000 INR) is set to exactly 
20,000. This collapses a small, widely spread set of extreme values into a single point, 
which is why we see a visible spike at 20,000 in the "After Capping" histogram — it's the 
visual signature of the capping operation itself.

## Transform data to obtain RFM

In [ ]:
snapshot_date = df['TransactionDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'TransactionDate': 'max',
    'TransactionID': 'count',   
    'TransactionAmount (INR)': 'sum'       
})

rfm['Recency'] = (snapshot_date - rfm['TransactionDate']).dt.days

rfm.rename(columns={
    'TransactionID': 'Frequency',
    'TransactionAmount (INR)': 'Monetary'
}, inplace=True)
rfm.head()

## RFM Customer Segments

Champions: Our best customers. They buy often and spend a lot.

Loyal Customers: Regular shoppers. They like our sales.

Potential Loyalists: New and active. They have room to grow.

Recent Customers: New arrivals. Just made their first purchase.

Promising: Good recent buyers. Just starting to spend.

Needs Attention: Used to spend a lot. Now quiet. Needs a reminder.

About to Sleep: Getting less active. Reach out to them soon.

At Risk: Long time since last buy. Urgent help needed.

Can’t Lose Them: Very loyal in the past. Now inactive for a while.

Hibernating: Not active anymore. Likely to be lost.

In [ ]:
def qcut_score(series, q, labels):
    """Quintile-score a series, preserving exact ties whenever qcut allows it.

    pd.qcut requires unique bin edges. Most RFM metrics have enough distinct values
    for this to just work — but Frequency in this dataset only has ~6 distinct values
    dataset-wide (see README limitations), which reliably breaks plain qcut. In that
    case only, we fall back to rank-based tie-breaking (ties get an arbitrary but
    valid order) so the pipeline doesn't crash. Recency and Monetary are expected to
    have enough distinct values to avoid the fallback, so their ties stay meaningful
    (identical values get identical scores) rather than being split arbitrarily.
    """
    try:
        return pd.qcut(series, q=q, labels=labels).astype(int)
    except ValueError:
        print(f"'{series.name}': too many repeated values for exact quantile binning "
              f"— falling back to rank-based tie-breaking.")
        return pd.qcut(series.rank(method='first'), q=q, labels=labels).astype(int)


rfm['R_Score'] = qcut_score(rfm['Recency'], 5, range(5, 0, -1))
rfm['F_Score'] = qcut_score(rfm['Frequency'], 5, range(1, 6))
rfm['M_Score'] = qcut_score(rfm['Monetary'], 5, range(1, 6))

rfm.head()

In [ ]:
segment_map = {}
for r in range(1, 6):
    for f in range(1, 6):
        if r >= 4 and f >= 4:
            seg = 'Champions'
        elif r == 3 and f >= 3:
            seg = 'Loyal Customers'
        elif r >= 4 and f == 3:
            seg = 'Potential Loyalists'
        elif r >= 4 and f <= 2:
            seg = 'Recent Customers'
        elif r == 3 and f <= 2:
            seg = 'Promising'
        elif r == 2 and f == 3:
            seg = 'Needs Attention'
        elif r == 2 and f <= 2:
            seg = 'About to Sleep'
        elif r == 1 and f == 3:
            seg = 'At Risk'
        elif r <= 2 and f >= 4:
            seg = "Can't Lose Them"
        else:
            seg = 'Hibernating'
        segment_map[(r, f)] = seg
        
rfm['Segment'] = rfm.apply(lambda row: segment_map[(row['R_Score'], row['F_Score'])], axis=1)
rfm['Segment'].value_counts()

**Note on methodology:** segments are assigned using only `R_Score` and `F_Score`. `M_Score` is computed and used later (see the R×F heatmap and the pairplot) to validate and describe the segments, but it does not directly determine segment membership. This is a common simplification in RFM segmentation grids, but it means a customer could in principle land in "Champions" with a comparatively low Monetary value — the heatmap below shows this is rare in practice, since Monetary rises sharply for high-Frequency customers, but it's worth keeping in mind when interpreting segment labels as guarantees of spend.

In [ ]:
segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=segment_counts.values, 
            y=segment_counts.index, 
            hue=segment_counts.index,
            legend=False,
            palette='viridis')
plt.title('Number of Customers per Segment')
plt.xlabel('Customer Count')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()

This chart shows how bank customers are spread across RFM segments: Champions are the biggest and most valuable group (~170,000), followed by Recent Customers (~133,000) who have good potential; the Can't Lose Them segment (~113,000) is concerning because these were high-value customers who are now becoming inactive, while Loyal Customers (~106,000) form a solid base; About to Sleep and Hibernating together add up to nearly 150,000 customers who are drifting away or already inactive and need re-engagement; Promising and Potential Loyalists (~65,000–70,000) could move up to better segments with the right campaigns; and although At Risk and Needs Attention are the smallest groups (~37,000 each), they still need quick action to prevent losing these customers completely.

**Note:** segment sizes are not purely a reflection of customer behavior — they are also shaped by how many (R_Score, F_Score) grid cells map to each segment. Champions, Recent Customers, and Can't Lose Them each cover 4 of the 25 grid cells, while At Risk and Needs Attention cover only 1 cell each, which partly explains why they are the smallest groups regardless of the underlying distribution.

In [ ]:
revenue_share = (rfm.groupby('Segment')['Monetary'].sum() / rfm['Monetary'].sum() * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=revenue_share.values,
    y=revenue_share.index, 
    hue=revenue_share.index, 
    legend=False, 
    palette='magma')
plt.title('Revenue Share by Segment (%)')
plt.xlabel('Share of Total Revenue (%)')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()

Looking at this chart alongside the first one gives a clearer picture: Champions are both the largest group in customer count (~170,000) and the top revenue contributor (26%), so they're truly the bank's most valuable segment overall. More interesting is Can't Lose Them — even though it has fewer customers than Recent Customers, it actually generates more revenue (13.5% vs 12.7%), meaning each customer in this group spends a lot on average, so losing them would hurt revenue more than their numbers suggest, making them a top priority for win-back efforts. On the other hand, About to Sleep and Hibernating together make up a large chunk of customers (~150,000) but contribute only about 7% each to revenue, showing these customers were never big spenders and are now becoming inactive — so re-engagement resources are better focused on high-value segments like Can't Lose Them and Loyal Customers rather than these low-value, fading groups.

In [ ]:
pivot = rfm.pivot_table(index='F_Score', columns='R_Score', values='Monetary', aggfunc='mean')

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, 
            annot=True, 
            fmt='.0f', 
            cmap='YlOrRd', 
            cbar_kws={'label': 'Avg. Monetary (INR)'})
plt.title('Average Monetary Value by R_Score and F_Score')
plt.xlabel('R_Score (5 = Most Recent)')
plt.ylabel('F_Score (5 = Most Frequent)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

This heatmap reveals a clear pattern: average Monetary value stays relatively flat 
(~1500–1650 INR) across F_Score levels 1 through 4, regardless of R_Score. The only row 
that stands out is F_Score = 5, where average Monetary jumps sharply to 2400–3200 INR, 
and within that row, value increases further with R_Score (from 2386 at R=1 to 3226 at R=5).

This confirms the earlier finding on Frequency's limited discriminative power: since the 
vast majority of customers fall into F_Score 1–4 with nearly identical purchase behavior, 
Frequency only becomes a meaningful signal at the very top tier (F_Score = 5). The 
customers who are both frequent (F=5) and recent (R=5) stand out as the highest-value 
group — consistent with the "Champions" segment definition — while the rest of the customer 
base shows little variation in spending regardless of how their R and F scores differ.

In [ ]:
segment_profiles = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=segment_profiles.sort_values('Recency'), 
            x='Recency', y='Segment', ax=axes[0], palette='Blues_r', hue='Segment', legend=False)
axes[0].set_title('Average Recency by Segment')
axes[0].set_xlabel('Avg. Recency (Days)')
axes[0].set_ylabel('')

sns.barplot(data=segment_profiles.sort_values('Frequency', ascending=False), 
            x='Frequency', y='Segment', ax=axes[1], palette='Greens_r', hue='Segment', legend=False)
axes[1].set_title('Average Frequency by Segment')
axes[1].set_xlabel('Avg. Frequency (Count)')
axes[1].set_ylabel('')

sns.barplot(data=segment_profiles.sort_values('Monetary', ascending=False), 
            x='Monetary', y='Segment', ax=axes[2], palette='Oranges_r', hue='Segment', legend=False)
axes[2].set_title('Average Monetary by Segment')
axes[2].set_xlabel('Avg. Monetary (INR)')
axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig("output/segment_profiles_dashboard.png", dpi=300, bbox_inches='tight')
plt.show()

This dashboard summarizes each segment's average Recency, Frequency, and Monetary values side by side, making it easy to see at a glance which segments are recent-but-low-spend versus long-inactive-but-historically-valuable, complementing the customer-count and revenue-share charts above.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=rfm, x='Recency', bins=30, ax=axes[0], color='skyblue')
axes[0].set_title('Recency Distribution (Days Since Last Transaction)')
axes[0].set_xlabel('Recency (Days)')
axes[0].set_ylabel('Number of Customers')

# Frequency only has ~6 distinct values dataset-wide, so the full range is already
# compact and readable — no percentile clipping needed (unlike Monetary, which is
# heavily right-skewed and would be unreadable without it).
sns.histplot(data=rfm, x='Frequency', discrete=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Frequency Distribution (Transaction Count)')
axes[1].set_xlabel('Frequency (Count)')
axes[1].set_ylabel('')

sns.histplot(data=rfm, x='Monetary', bins=50, ax=axes[2], color='salmon',
             binrange=(0, rfm['Monetary'].quantile(0.99)))
axes[2].set_title('Monetary Distribution (Total Spend per Customer)')
axes[2].set_xlabel('Monetary (INR)')
axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig("output/rfm_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
rfm_sample = rfm.sample(n=20000, random_state=42)

sns.pairplot(rfm_sample, vars=['Recency', 'Frequency', 'Monetary'], 
             hue='Segment', palette='tab10', 
             plot_kws={'alpha': 0.4, 's': 15},
             diag_kind='kde', height=3)
plt.suptitle('RFM Relationships by Segment (20k sample)', y=1.02)
plt.savefig("output/rfm_pairplot.png", dpi=300, bbox_inches='tight')
plt.show()

## Recommended Actions by Segment

In [ ]:
segment_summary = pd.DataFrame({
    'Customer_Count': segment_counts,
    'Revenue_Share_Pct': revenue_share.round(1)
}).sort_values('Revenue_Share_Pct', ascending=False)

segment_summary

**Takeaway:** Champions and Can't Lose Them together represent under a third of customers but generate close to 40% of revenue, making them the clear priority for retention spending. About to Sleep and Hibernating make up a large customer share (~17%) but contribute comparatively little revenue, so re-engagement resources are better directed toward the high-value, at-risk segments (Can't Lose Them, Needs Attention) than toward reactivating low-spending inactive customers.

| Segment | Recommended Action |
|---|---|
| Champions | Reward with loyalty perks; use as referral source |
| Can't Lose Them | Priority win-back campaign; personal outreach |
| Loyal Customers | Maintain engagement with regular promotions |
| Recent Customers | Onboarding push to encourage a second purchase |
| Promising / Potential Loyalists | Targeted campaigns to move them toward Loyal/Champion status |
| About to Sleep / Hibernating | Low-cost re-engagement (email); avoid over-investing |
| At Risk / Needs Attention | Quick, low-cost check-in to prevent full churn |

*(Exact customer counts and revenue shares are computed programmatically in the cell above — reusing the same `segment_counts` and `revenue_share` used for the charts — rather than hardcoded or recomputed separately, so this table can't drift out of sync with the charts.)*

In [ ]:
rfm.to_csv('output/rfm_segments.csv')